# 01 · Exploración del Voronoi (corte del 14 de octubre)

**Qué hace.** Responde si el control territorial (Voronoi) distingue al América de Jardine del de Solari y Ortiz, y si se relaciona con el rendimiento, para decidir el 14 de octubre si sigue como lente central.

**Qué necesita.** Los pasos 02–04 corridos: `data/processed/` (03) y `outputs/voronoi/metricas/` (04).

**Cómo probarlo.** En VS Code, elige el kernel `.venv` y pulsa *Run All*. Tarda 1–2 minutos. Las figuras se guardan en `outputs/exploracion/` (local, no se suben).

**Qué responde.**
1. ¿Las diferencias de control entre entrenadores son reales o ruido? (intervalos por bootstrap de partidos)
2. ¿Dónde gana o pierde espacio el América con Jardine? (mapa por zona)
3. ¿Sigue la diferencia si se compara con la misma cobertura de cámara? (regresión con jugadores visibles)
4. ¿El control se relaciona con el xG del partido? (correlación por partido)

**Definiciones.** *Control*: fracción del espacio visible y alcanzado (radio de 20 yd) que controla el equipo que ejecuta la acción (`contexto/framework.md`). *Con balón*: el equipo del evento. *Tercio*: en la perspectiva del equipo con balón (propio = su tercio defensivo). Solo juego abierto. La unidad de análisis es el **partido**, no la acción.

> IMPOR1: solo se muestran agregados (promedios por entrenador, zona o partido). El notebook se sube sin salidas (nbstripout).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from matplotlib.colors import LinearSegmentedColormap
from mplsoccer import Pitch

from ojo.config import DIR_SALIDAS, params
from ojo.datos import almacen
from ojo.framework import juego
from ojo.framework import partidos as fp
from ojo.salidas.estilo import color, rgba

P = params()
EQUIPO = P["equipo"]["team_id"]
EST = P["estadistica"]
RNG = np.random.default_rng(P["simulacion"]["semilla"])
DIR_FIG = DIR_SALIDAS / "exploracion"
DIR_FIG.mkdir(parents=True, exist_ok=True)

# Nombre corto de cada entrenador del análisis (ids en params.yaml), en orden cronológico
ENTRENADORES = {d["manager_id"]: d["nombre"].split()[-1] for d in P["predecesores"]}
ENTRENADORES[P["entrenador"]["manager_id"]] = P["entrenador"]["nombre"].split()[-1]
ORDEN = list(ENTRENADORES.values())
NUEVO, PREVIOS = ORDEN[-1], ORDEN[:-1]
COLORES = {ORDEN[0]: color("gris-texto"), ORDEN[1]: color("azul"), NUEVO: color("dorado")}
ORDEN

## Datos

Se unen las métricas de Voronoi por evento con la ubicación del evento y el entrenador del América en ese partido.

In [ ]:
metricas = pd.concat([pd.read_parquet(f) for f in (DIR_SALIDAS / "voronoi" / "metricas").glob("*.parquet")],
                     ignore_index=True)
eventos = almacen.cargar("eventos", columnas=["id", "match_id", "x", "y", "patron"])
partidos = fp.con_torneo(fp.del_equipo(almacen.cargar("partidos"), EQUIPO))[["match_id", "entrenador_id", "torneo_corto"]]

datos = metricas.merge(eventos, on=["id", "match_id"]).merge(partidos, on="match_id")
datos = datos[juego.es_juego_abierto(datos) & datos["entrenador_id"].isin(ENTRENADORES)].copy()
datos["dt"] = pd.Categorical(datos["entrenador_id"].map(ENTRENADORES), categories=ORDEN, ordered=True)
datos["con_balon"] = np.where(datos["equipo_id"] == EQUIPO, "America", "Rival")
t = P["tercios"]
datos["tercio"] = pd.cut(datos["x"], [-np.inf, t["defensivo_max_x"], t["final_min_x"], np.inf],
                         labels=["propio", "medio", "final"])
print(f"{len(datos):,} acciones en juego abierto con Voronoi · {datos['match_id'].nunique()} partidos")
datos.groupby(["dt", "con_balon"], observed=True)["match_id"].nunique().unstack().rename_axis(None, axis=1)

In [ ]:
def por_partido(df: pd.DataFrame, valor: str, grupos: list[str]) -> pd.DataFrame:
    # Promedio por partido: cada partido pesa lo mismo, tenga muchas o pocas acciones.
    return df.groupby(["match_id", *grupos], observed=True)[valor].mean().reset_index()


def ic_bootstrap(valores, repeticiones=EST["bootstrap_repeticiones"], nivel=EST["nivel_confianza"]):
    # Media e intervalo de confianza remuestreando partidos.
    v = np.asarray(valores, dtype=float)
    v = v[~np.isnan(v)]
    medias = RNG.choice(v, size=(repeticiones, len(v)), replace=True).mean(axis=1)
    q = (1 - nivel) / 2
    return v.mean(), *np.quantile(medias, [q, 1 - q])


def ic_diferencia(a, b, repeticiones=EST["bootstrap_repeticiones"], nivel=EST["nivel_confianza"]):
    # Diferencia de medias (a − b) con intervalo, remuestreando partidos de cada grupo por separado.
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    d = RNG.choice(a, (repeticiones, len(a))).mean(axis=1) - RNG.choice(b, (repeticiones, len(b))).mean(axis=1)
    q = (1 - nivel) / 2
    return a.mean() - b.mean(), *np.quantile(d, [q, 1 - q])

## 1 · ¿Las diferencias entre entrenadores son reales?

Control promedio por partido, por tercio y por quién tiene el balón, con su intervalo de confianza. Después, la diferencia de Jardine contra cada predecesor: si el intervalo no cruza el cero, la diferencia es distinguible del ruido.

In [ ]:
pp = por_partido(datos, "control", ["dt", "con_balon", "tercio"])
filas = []
for (dt, balon, tercio), g in pp.groupby(["dt", "con_balon", "tercio"], observed=True):
    media, bajo, alto = ic_bootstrap(g["control"])
    filas.append({"con_balon": balon, "tercio": tercio, "dt": dt, "partidos": len(g),
                  "control": media, "ic_bajo": bajo, "ic_alto": alto})
ic = pd.DataFrame(filas)
ic.pivot_table(index=["con_balon", "tercio"], columns="dt", values="control", observed=True).round(3)

In [ ]:
dif = []
for (balon, tercio), g in pp.groupby(["con_balon", "tercio"], observed=True):
    for previo in PREVIOS:
        d, bajo, alto = ic_diferencia(g.loc[g["dt"] == NUEVO, "control"], g.loc[g["dt"] == previo, "control"])
        dif.append({"con_balon": balon, "tercio": tercio, "comparacion": f"{NUEVO} − {previo}",
                    "diferencia_pp": 100 * d, "ic_bajo_pp": 100 * bajo, "ic_alto_pp": 100 * alto,
                    "distinguible": bool(bajo > 0 or alto < 0)})
diferencias = pd.DataFrame(dif)
diferencias.round(2)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for eje, balon in zip(ejes, ["America", "Rival"]):
    sub = ic[ic["con_balon"] == balon]
    for i, dt in enumerate(ORDEN):
        s = sub[sub["dt"] == dt]
        x = np.arange(len(s)) + (i - 1) * 0.22
        eje.errorbar(x, s["control"], yerr=[s["control"] - s["ic_bajo"], s["ic_alto"] - s["control"]],
                     fmt="o", color=COLORES[dt], capsize=3, label=dt)
    eje.set_xticks(range(3), ["Propio", "Medio", "Final"])
    eje.set_title(f"{'El América' if balon == 'America' else 'El rival'} con el balón")
    eje.axhline(0.5, color=rgba("gris-texto", 0.4), lw=1, ls="--")
ejes[0].set_ylabel("Control del equipo con balón")
ejes[0].legend(frameon=False)
fig.suptitle("Control del espacio visible por tercio (promedio por partido, IC 95 %)", fontweight="bold")
fig.tight_layout()
fig.savefig(DIR_FIG / "01_control_por_tercio.png", dpi=130)

## 2 · ¿Dónde gana o pierde espacio el América con Jardine?

Control promedio del América con balón según la zona de la cancha donde ocurre la acción (rejilla de `params.yaml`), y la resta Jardine − predecesor inmediato. Zonas con menos de `estadistica.min_eventos_zona` acciones quedan en blanco. Es un mapa descriptivo: no lleva intervalos por zona.

In [ ]:
REJ, CAMPO = P["resumenes"]["rejilla"], P["campo"]


def mapa_control(df: pd.DataFrame) -> pd.DataFrame:
    # Control promedio por zona (filas = y, columnas = x) donde ocurre la acción.
    col = np.clip((df["x"] / CAMPO["largo"] * REJ["columnas"]).astype(int), 0, REJ["columnas"] - 1)
    fil = np.clip((df["y"] / CAMPO["ancho"] * REJ["filas"]).astype(int), 0, REJ["filas"] - 1)
    g = df.assign(col=col, fil=fil).groupby(["fil", "col"])["control"].agg(["mean", "size"])
    indice = dict(index=range(REJ["filas"]), columns=range(REJ["columnas"]))
    media = g["mean"].unstack().reindex(**indice)
    n = g["size"].unstack().reindex(**indice).fillna(0)
    return media.where(n >= EST["min_eventos_zona"])


america = datos[datos["con_balon"] == "America"]
mapas = {dt: mapa_control(america[america["dt"] == dt]) for dt in ORDEN}
diferencia_zona = 100 * (mapas[NUEVO] - mapas[PREVIOS[-1]])
print(f"Diferencia por zona ({NUEVO} − {PREVIOS[-1]}), puntos: mín {np.nanmin(diferencia_zona.values):.1f}, "
      f"máx {np.nanmax(diferencia_zona.values):.1f}, mediana {np.nanmedian(diferencia_zona.values):.1f}")

In [ ]:
divergente = LinearSegmentedColormap.from_list("oda", [color("azul"), color("blanco"), color("dorado")])
pitch = Pitch(pitch_type="statsbomb", line_color=color("gris-texto"), line_zorder=3)
fig, ejes = pitch.draw(ncols=2, figsize=(14, 5))
for eje, (titulo, valores, cmap, vmin, vmax) in zip(ejes, [
        (f"Control con balón · {NUEVO}", mapas[NUEVO], LinearSegmentedColormap.from_list("d", [color("blanco"), color("dorado")]), 0.3, 0.7),
        (f"{NUEVO} − {PREVIOS[-1]} (puntos de control)", diferencia_zona, divergente, -8, 8)]):
    im = eje.imshow(valores.values, extent=[0, CAMPO["largo"], CAMPO["ancho"], 0], cmap=cmap, vmin=vmin, vmax=vmax,
                    zorder=1, alpha=0.9)
    eje.set_title(titulo)
    fig.colorbar(im, ax=eje, shrink=0.7)
fig.suptitle("El América ataca hacia la derecha · zonas en blanco: pocas acciones", y=0.98)
fig.savefig(DIR_FIG / "02_mapa_control.png", dpi=130, bbox_inches="tight")

## 3 · ¿Sigue la diferencia con la misma cobertura de cámara?

Con Jardine la cámara ve más jugadores por frame, y eso cambia el control. Se estima el efecto de cada entrenador **descontando** el número de jugadores visibles y el tercio, con errores estándar agrupados por partido (las acciones de un mismo partido no son independientes). Referencia: Jardine; un coeficiente positivo significa que ese predecesor tenía más control que Jardine.

In [ ]:
formula = f"control ~ C(dt, Treatment(reference='{NUEVO}')) + n_jugadores + C(tercio)"
efectos = []
for balon in ["America", "Rival"]:
    sub = datos[datos["con_balon"] == balon].dropna(subset=["control"])
    sub = sub.assign(dt=sub["dt"].astype(str), tercio=sub["tercio"].astype(str))
    modelo = smf.ols(formula, data=sub).fit(cov_type="cluster", cov_kwds={"groups": sub["match_id"]})
    ic95 = modelo.conf_int(alpha=1 - EST["nivel_confianza"])
    for termino in modelo.params.index:
        if termino.startswith("C(dt") or termino == "n_jugadores":
            efectos.append({"con_balon": balon, "termino": termino.split("[T.")[-1].rstrip("]"),
                            "efecto_pp": 100 * modelo.params[termino], "ic_bajo_pp": 100 * ic95.loc[termino, 0],
                            "ic_alto_pp": 100 * ic95.loc[termino, 1], "p": modelo.pvalues[termino]})
pd.DataFrame(efectos).round(3)

### 3b · ¿Y las métricas del grafo de vecindad?

El control es solo una de las métricas del Voronoi. Mismo modelo (descontando jugadores visibles y tercio, errores por partido) para las métricas del grafo: compañeros cerca del que tiene el balón (*apoyos*), rivales encima (*rivales_cerca*), distancia entre compañeros vecinos (*compactación*) y largo de la *frontera*. Se reporta el efecto de cada predecesor frente a Jardine en la unidad de la métrica.

In [ ]:
grafo = []
for metrica in ["apoyos", "rivales_cerca", "distancia_companeros", "frontera"]:
    for balon in ["America", "Rival"]:
        sub = datos[datos["con_balon"] == balon].dropna(subset=[metrica])
        sub = sub.assign(dt=sub["dt"].astype(str), tercio=sub["tercio"].astype(str))
        modelo = smf.ols(f"{metrica} ~ C(dt, Treatment(reference='{NUEVO}')) + n_jugadores + C(tercio)",
                         data=sub).fit(cov_type="cluster", cov_kwds={"groups": sub["match_id"]})
        ic95 = modelo.conf_int(alpha=1 - EST["nivel_confianza"])
        media_nuevo = sub.loc[sub["dt"] == NUEVO, metrica].mean()
        for termino in modelo.params.index:
            if termino.startswith("C(dt"):
                grafo.append({"metrica": metrica, "con_balon": balon, "predecesor": termino.split("[T.")[-1].rstrip("]"),
                              f"media_{NUEVO}": media_nuevo, "efecto": modelo.params[termino],
                              "ic_bajo": ic95.loc[termino, 0], "ic_alto": ic95.loc[termino, 1], "p": modelo.pvalues[termino],
                              "distinguible": bool(ic95.loc[termino, 0] > 0 or ic95.loc[termino, 1] < 0)})
pd.DataFrame(grafo).round(3)

## 4 · ¿El control se relaciona con el resultado?

Por partido: control del América con balón en el tercio final frente a su xG sin penales, y la lectura a dos bandas (control del América con balón − control del rival con balón) frente a la diferencia de xG. Correlación de Spearman con intervalo por bootstrap. **Es correlación, no causalidad** (regla 6).

In [ ]:
tms = almacen.cargar("estadisticas_equipo", match_ids=partidos["match_id"],
                     columnas=["match_id", "team_id", "team_match_np_xg", "team_match_np_xg_conceded"])
xg = tms[tms["team_id"] == EQUIPO].set_index("match_id")
rival = datos[datos["con_balon"] == "Rival"]
por_partido_tabla = pd.DataFrame({
    "control_final": america[america["tercio"] == "final"].groupby("match_id")["control"].mean(),
    "dos_bandas": america.groupby("match_id")["control"].mean() - rival.groupby("match_id")["control"].mean(),
}).join(xg)
por_partido_tabla["xg_dif"] = por_partido_tabla["team_match_np_xg"] - por_partido_tabla["team_match_np_xg_conceded"]
por_partido_tabla = por_partido_tabla.dropna()


def spearman_ic(x, y, repeticiones=EST["bootstrap_repeticiones"], nivel=EST["nivel_confianza"]):
    x, y = np.asarray(x), np.asarray(y)
    rho = pd.Series(x).corr(pd.Series(y), method="spearman")
    idx = RNG.integers(0, len(x), size=(repeticiones, len(x)))
    boot = [pd.Series(x[i]).corr(pd.Series(y[i]), method="spearman") for i in idx]
    q = (1 - nivel) / 2
    return rho, *np.quantile(boot, [q, 1 - q])


relaciones = []
for x, y in [("control_final", "team_match_np_xg"), ("dos_bandas", "xg_dif")]:
    rho, bajo, alto = spearman_ic(por_partido_tabla[x], por_partido_tabla[y])
    relaciones.append({"control": x, "resultado": y, "partidos": len(por_partido_tabla),
                       "spearman": rho, "ic_bajo": bajo, "ic_alto": alto})
pd.DataFrame(relaciones).round(3)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.4))
for eje, (x, y, etiqueta_x, etiqueta_y) in zip(ejes, [
        ("control_final", "team_match_np_xg", "Control del América con balón en el tercio final", "xG del América (sin penales)"),
        ("dos_bandas", "xg_dif", "Control América − control rival (con balón)", "Diferencia de xG")]):
    eje.scatter(por_partido_tabla[x], por_partido_tabla[y], s=18, color=color("dorado"), edgecolor=color("azul"), lw=0.5)
    pendiente, ordenada = np.polyfit(por_partido_tabla[x], por_partido_tabla[y], 1)
    xs = np.linspace(por_partido_tabla[x].min(), por_partido_tabla[x].max(), 50)
    eje.plot(xs, pendiente * xs + ordenada, color=color("azul"), lw=2)
    eje.set_xlabel(etiqueta_x)
    eje.set_ylabel(etiqueta_y)
fig.suptitle("Cada punto es un partido del América (AP21–CL26)", fontweight="bold")
fig.tight_layout()
fig.savefig(DIR_FIG / "04_control_y_xg.png", dpi=130)

## Lectura para el corte del 14 de octubre

Primera corrida (12 oct, 210 partidos, 387,897 acciones en juego abierto). Si los datos cambian, vuelve a correr y revisa que los números sigan diciendo lo mismo.

1. **El control promedio casi no separa entrenadores.** En crudo hay diferencias de 1–2 puntos (p. ej., en su propio tercio el América controla 1.6 puntos menos con Jardine que con Solari), pero **al descontar los jugadores visibles y el tercio, ninguna diferencia de entrenador es distinguible del ruido** (sección 3). Cada jugador visible de más baja el control ≈ 0.2 puntos: buena parte de la diferencia cruda era la cámara.
2. **Las métricas del grafo tampoco dan un hallazgo de cancha.** Algunas diferencias son estadísticamente distinguibles por el tamaño de la muestra (p. ej., la frontera es ≈ 2 yd más corta con Jardine, sobre ≈ 143), pero son de 1–3 %: demasiado pequeñas para que un cuerpo técnico las note.
3. **Por zona** las diferencias Jardine − Ortiz van de −3.4 a +3.7 puntos, con mediana −0.4: sin un patrón claro (y sin intervalos por zona).
4. **El control sí se relaciona, débil, con el rendimiento.** Más control en el tercio final va con más xG (Spearman 0.21, IC 95 % 0.08–0.33). La lectura a dos bandas promedio no se relaciona con la diferencia de xG (−0.01). Es correlación, no causalidad.

**Qué implica.** Según la regla de corte, el Voronoi **no da un hallazgo claro para comparar entrenadores** con estos promedios: para eso pasaría a lente de apoyo. Sigue siendo útil como lenguaje visual (interpretabilidad) y como variable con señal débil en la parte predictiva. Antes de cerrar la decisión conviene revisar dónde aún podría marcar diferencia: por situación (salida, presión alta, marcador), por rival concreto y tramo a tramo dentro del partido. Esa es la pregunta del explorador, no la del promedio del entrenador.